In [ ]:
%run GeneralTaxiTransformer

In [ ]:
%run geo_zones

In [ ]:
from pyspark.sql.functions import *

In [ ]:
class YellowTaxiTransformer(GeneralTaxiTransformer):

    TABLE_NAME = "yellow_tripdata"

    ADDITIONAL_SCHEMA = {
        "passenger_count": {"type": "integer", "nullable": True},
        "trip_distance": {"type": "double", "nullable": True},
        "pickup_longitude": {"type": "double", "nullable": True},
        "pickup_latitude": {"type": "double", "nullable": True},
        "dropoff_longitude": {"type": "double", "nullable": True},
        "dropoff_latitude": {"type": "double", "nullable": True},
        "fare_amount": {"type": "double", "nullable": True},
        "total_amount": {"type": "double", "nullable": True},
        "tip_amount": {"type": "double", "nullable": True}
    }
    ADDITIONAL_RESULT_COLUMNS = ["passenger_count", "trip_distance", "fare_amount", "total_amount", "tip_amount"]
    ADDITIONAL_COLUMN_MAPPING = {
        "Passenger_Count": "passenger_count",
        "Trip_Distance": "trip_distance",
        "Start_Lon": "pickup_longitude",
        "Start_Lat": "pickup_latitude",
        "End_Lon": "dropoff_longitude",
        "End_Lat": "dropoff_latitude",
        "Fare_Amt": "fare_amount",
        "Total_Amt": "total_amount",
        "Tip_Amt": "tip_amount"
    }

    TAXI_VALUE_RANGES = {
        "trip_distance": (0, None),
        "fare_amount": (0, None),
        "total_amount": (0, None),
        "tip_amount": (0, None)
    }

    def __init__(self):
        super().__init__()

    def is_valid_taxi_location(self, location_id, longitude, latitude):
        return (col(location_id).isNotNull() | ((col(longitude) != 0) & (col(latitude) != 0)))

    def filter_taxi_coordinates(self, df):
        is_pickup_valid = self.is_valid_taxi_location("pickup_location_id", "pickup_longitude", "pickup_latitude")
        is_dropoff_valid = self.is_valid_taxi_location("dropoff_location_id", "dropoff_longitude", "dropoff_latitude")
        return df.filter(is_pickup_valid | is_dropoff_valid)

    def prepare_sedona_env(self, spark, shapefile = None):
        return prepare_spatial_processing(spark)

    def enrich_location(self, df, sedona):
        return add_zone_from_coordinates_or_location(df, sedona)
    
    def assign_special_location_id(self, df, location_id, longitude, latitude, unknown_location_id, outside_nyc_location_id):
        return df.withColumn(location_id,
            when(
                col(location_id).isNull() &
                ((col(longitude) == 0.0) | (col(latitude) == 0.0)),
                unknown_location_id
            ).when(
                col(location_id).isNull() &
                (col(longitude) != 0.0) & (col(latitude) != 0.0),
                outside_nyc_location_id
            ).otherwise(col(location_id))
        )
    
    def assign_special_location_ids(self, df, unknown_location_id, outside_nyc_location_id):
        for coor_type in ["pickup", "dropoff"]:
            df = self.assign_special_location_id(df, f"{coor_type}_location_id", f"{coor_type}_longitude", f"{coor_type}_latitude", unknown_location_id, outside_nyc_location_id)
        return df
    